# 03 · BERTweet (texte seul)

Fine-tuning de `vinai/bertweet-base` sur le texte des tweets de CrisisMMD, protocole A (split officiel « agreed labels »).

Pour l'instant le notebook lit les TSV officiels bruts. Il passera sur les splits nettoyés (`src/data.py`) dès qu'ils seront sur `main`.

In [ ]:
from pathlib import Path

try:  # Colab : dossier partagé du Drive
    from google.colab import drive

    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/DATA4001")
    %pip install -q emoji
except ImportError:  # en local : racine du repo, dépendances de requirements.txt
    ROOT = Path("..")

DATA_DIR = ROOT / "data" / "crisismmd_datasplit_agreed_label"

In [ ]:
import csv

import pandas as pd
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, DataCollatorWithPadding

TASK = "informative"  # tâche 1 d'abord, puis "humanitarian"
MODEL_NAME = "vinai/bertweet-base"
MAX_LEN = 128
BATCH_SIZE = 32

## Données

In [ ]:
def load_split(split):
    path = DATA_DIR / f"task_{TASK}_text_img_agreed_lab_{split}.tsv"
    # QUOTE_NONE : ces TSV n'échappent rien, sinon pandas retire les guillemets des tweets qui commencent par "
    return pd.read_csv(path, sep="\t", quoting=csv.QUOTE_NONE)


train, dev, test = (load_split(split) for split in ("train", "dev", "test"))
train = train.drop_duplicates("tweet_id")  # modèle texte : un exemple par tweet, pas par image

classes = sorted(train["label"].unique())
label2id = {label: i for i, label in enumerate(classes)}

print(f"train : {len(train)} tweets · dev : {len(dev)} · test : {len(test)}")
train["label"].value_counts()

## Tokenisation

In [ ]:
# normalization=True applique la normalisation de BERTweet : mentions -> @USER, URLs -> HTTPURL, emojis -> texte
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, normalization=True)


def make_loader(df, shuffle=False):
    encodings = tokenizer(df["tweet_text"].tolist(), truncation=True, max_length=MAX_LEN)
    examples = [
        {"input_ids": ids, "attention_mask": mask, "labels": label2id[label]}
        for ids, mask, label in zip(encodings["input_ids"], encodings["attention_mask"], df["label"])
    ]
    # padding dynamique : chaque batch est complété à la longueur de son tweet le plus long
    return DataLoader(examples, batch_size=BATCH_SIZE, shuffle=shuffle, collate_fn=DataCollatorWithPadding(tokenizer))


train_loader, dev_loader, test_loader = make_loader(train, shuffle=True), make_loader(dev), make_loader(test)

print(train["tweet_text"].iloc[0])
print(tokenizer.decode(train_loader.dataset[0]["input_ids"]))